# Phase 2 driver

A **driver only** (CLAUDE2.md "Code layout"). Every cell here imports `src/`
code to launch a run or look at a result. No loader, training loop, config
system or baseline logic is defined in this notebook — that lives in `src/`
with tests and is run through `python -m scripts.t20x_*`.

Run `jupyter lab` from the repo root. Strip outputs before committing:
`nbstripout notebooks/phase2.ipynb`.

In [ ]:
import os
import pathlib
import sys

# Work from the repo root so `import src...` and the anchored corpus paths resolve
# whether the kernel started here or in notebooks/.
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

In [ ]:
# T-201 — environment guard. Same check every entry point runs.
from src.env_check import check_env

check_env()

## T-202 — data loading

`src/data.py`. Native splits only for training (`hard rule 1`); `load_split`
asserts the origin so a machine-translated split can't slip in.

In [ ]:
from src import data as D

TASK = 2  # 2 = sustainability (binary), 3 = ESG themes (10-class)
frame = D.load_native(TASK, "hin")
print(len(frame), "rows", "| labels:", D.label_names(TASK))
frame["label"].value_counts(normalize=True)

In [ ]:
# Seeded, class-stratified train/dev/test. Same seed -> identical partition.
parts = D.stratified_split(frame, seed=0, dev=0.1, test=0.1)
{name: len(part) for name, part in parts.items()}

In [ ]:
# Tokenise a split and look at one padded batch. mBERT here because it needs no
# gated-repo access; swap for 'indicbert-v2' once its terms are accepted.
ds = D.load_dataset(TASK, "hin", "mbert-base", max_len=128)
batch = ds.collate([ds[i] for i in range(8)])
{k: (v.shape if hasattr(v, "shape") else v) for k, v in batch.items()}

## T-203 — training loop

Fill in once `src/train.py` exists. Driver only: build a model + the datasets
above, call the training entry point, plot the dev-F1 curve it returns. The
50-example overfit check runs as `python -m scripts.t203_smoke` / a test, not
here.

## T-205 — VRAM budgets

Probe max `batch × max_len` per encoder on the 4 GB card here — interactive
trial and error is what a notebook is good for. Record the resulting table in
`reports/`, not in this notebook.

## T-207 — baseline gap

Load `experiments.csv`, compare the Hindi macro-F1 to IndicFinNLP's published
number, inspect misclassified rows to diagnose any gap before touching
hyperparameters.